# 00_setup — Colab Environment Setup (Clinical Trust KG-RAG)

This notebook prepares the Colab environment for the full pipeline: mounts Drive, confirms the dataset is present, loads Neo4j Aura credentials, installs FAISS, installs and starts Ollama, pulls the `deepseek-r1:7b` model, and installs all remaining Python packages.

**Run every cell top-to-bottom, in order, in a single session.** Do not skip cells or re-run out of sequence — that is what caused the previous failures (`ollama pull` before `ollama serve`, and `faiss-gpu` silently breaking the combined install line).

**Before running:** go to `Runtime` → `Change runtime type` → select **T4 GPU**, and make sure you've added `NEO4J_URI` and `NEO4J_PASSWORD` as Colab Secrets (key icon on the left sidebar) with **Notebook access** turned on.

## Step 1 — Mount Google Drive & create project folders

This connects Google Drive so files persist across Colab sessions, and creates the folder structure under `ClinicalTrust` if it doesn't already exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/ClinicalTrust"
for sub in ["data/raw", "data/processed", "models", "notebooks", "reports/baselines", "reports/evaluation"]:
    os.makedirs(os.path.join(PROJECT_ROOT, sub), exist_ok=True)
print("Drive mounted. Project root:", PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Project root: /content/drive/MyDrive/ClinicalTrust


## Step 2 — Confirm the dataset is present

Checks that `PMC-Patients.csv` is actually sitting in `data/raw` on Drive. This should print `True`. If it prints `False`, upload the CSV into that folder via the Drive web UI before continuing.

In [ ]:
csv_path = f"{PROJECT_ROOT}/data/raw/PMC-Patients.csv"
print("CSV found:", os.path.exists(csv_path))
print("Path:", csv_path)

CSV found: True
Path: /content/drive/MyDrive/ClinicalTrust/data/raw/PMC-Patients.csv


## Step 3 — Load Neo4j Aura credentials from Colab Secrets

Reads `NEO4J_URI` and `NEO4J_PASSWORD` from Colab's Secrets manager. Both should print `True`. This only checks that the secrets *loaded* — the actual live connection is verified later in Step 8, after the `neo4j` Python package is installed.

In [ ]:
from google.colab import userdata
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')
print("URI loaded:", bool(NEO4J_URI))
print("Username loaded:", bool(NEO4J_USERNAME))
print("Password loaded:", bool(NEO4J_PASSWORD))

URI loaded: True
Username loaded: True
Password loaded: True


## Step 4 — Install FAISS (CPU) on its own

`faiss-gpu` has no working wheel for Colab's current Python/CUDA and will fail to build. `faiss-cpu` is fast enough for this dataset size and works reliably.

**Important:** install this by itself, in its own cell — never combine it with other packages in one `pip install` line. If one package in a combined line fails to build, pip can abort the *entire* line, silently skipping every other package after it (this is exactly what happened before with `neo4j`, `transformers`, and `langgraph` never actually installing).

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss
print("FAISS version:", faiss.__version__)

FAISS version: 1.15.0


## Step 5 — Install system dependency (`zstd`) and Ollama

`zstd` is required for Ollama's install script to unpack correctly. Then the official Ollama install script is run.

In [ ]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 67 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Step 6 — Start the Ollama server, then pull `deepseek-r1:7b`

This is the step that broke before: `ollama pull` was run before the server was up. Ollama has to run as a **background process** first — it doesn't run as a normal foreground command in Colab. The cell below starts it, then actively waits (polling, not just a fixed `sleep`) until the server responds before pulling the model. The pull itself can take a couple of minutes (~4.7GB).

In [ ]:
import subprocess, time, requests

ollama_process = subprocess.Popen(["ollama", "serve"])

# Poll until the server actually responds, instead of guessing with a fixed sleep
for attempt in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not start in time — re-run this cell.")

Ollama server is up.


In [ ]:
!ollama pull deepseek-r1:7b

## Step 7 — Verify the model downloaded

Should list `deepseek-r1:7b` with a size around 4.7GB.

In [ ]:
!ollama list

NAME              ID              SIZE      MODIFIED               
deepseek-r1:7b    755ced02ce7b    4.7 GB    Less than a second ago    


## Step 8 — Install remaining Python packages

These packages are pure-Python / have pre-built wheels, so they install quickly and reliably together. `faiss-gpu` is deliberately **excluded** — it's already installed as `faiss-cpu` in Step 4 and must never appear in this line. `scispacy` and its biomedical model are installed **separately in the next two cells**, not here — see the note below.

In [ ]:
!pip install -q transformers neo4j langgraph python-dotenv rouge-score ollama

## Step 8b — Install scispaCy and the biomedical NER model

**This is what actually failed before.** The `en_core_sci_sm-0.5.4` model file is an old release that pins an old version of `spacy`. Colab's current Python has no pre-built wheel for that old spacy version, so pip tries to *compile it from source* — and that source build fails (`pip subprocess to install build dependencies did not run successfully`). Bundling it into the same line as `neo4j`/`transformers`/`langgraph` also silently blocked those from installing.

The fix: install `scispacy` by itself first, so it pulls in a `spacy` version that's actually compatible with Colab's Python. Then install the model file with `--no-deps`, so it reuses that already-installed compatible spacy instead of trying to force its own old pin.

In [ ]:
!pip install -q scispacy

  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Preparing metadata (setup.py) ... done


Verify the model actually loads before moving on — this is more reliable than trusting a silent pip success message.

In [ ]:
import en_core_sci_sm, os, glob, re

pkg_dir = os.path.dirname(en_core_sci_sm.__file__)
config_paths = glob.glob(os.path.join(pkg_dir, "**", "config.cfg"), recursive=True)

for path in config_paths:
    with open(path, "r") as f:
        content = f.read()
    fixed = re.sub(r'=\s*"True"', "= true", content)
    fixed = re.sub(r'=\s*"False"', "= false", fixed)
    if fixed != content:
        with open(path, "w") as f:
            f.write(fixed)
        print("Patched:", path)

nlp = en_core_sci_sm.load()
print("scispaCy model loaded OK:", nlp.meta["name"])

/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.15). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/usr/local/lib/python3.13/dist-packages/spacy/language.py:2235: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


scispaCy model loaded OK: core_sci_sm


## Step 9 — Final combined verification

Confirms the GPU is available and does a **live** query against Neo4j Aura (not just checking that the secrets loaded). If both lines print successfully, setup is complete and you can move on to `01_data_ingestion.ipynb`.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

from neo4j import GraphDatabase
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
with driver.session() as session:
    result = session.run("RETURN 'Connected to Aura!' AS message")
    print(result.single()["message"])
driver.close()

GPU available: True
GPU name: Tesla T4
Connected to Aura!


# New section

---
### If something fails
- **`ModuleNotFoundError: No module named 'neo4j'` (or transformers/langgraph):** Step 8 didn't complete — re-run Step 8's cell and check the output for errors before moving on.
- **`pip subprocess to install build dependencies did not run successfully` (when installing scispaCy or the model):** this means Step 8b is trying to compile an old spaCy version from source. Restart the runtime (`Runtime` → `Restart session`), re-run Steps 1–8, then run Step 8b's two cells in order — `scispacy` first, then the model with `--no-deps`. Don't run them out of order or skip the `--no-deps` flag.
- **`could not connect to ollama server`:** Step 6's first cell didn't finish starting the server — re-run it and wait for "Ollama server is up." before running the pull cell.
- **Neo4j connection error in Step 9:** double-check the `NEO4J_URI` and `NEO4J_PASSWORD` secrets are correct and that Notebook access is enabled for both, then re-run Step 3 and Step 9.
- **Colab disconnects/restarts:** must re-run this entire notebook from Step 1 — nothing installed in RAM (Ollama, pip packages) survives a runtime restart, only files saved to Drive do.